# Fine-Tuning Open Source ASR for Japanese

## A Comprehensive Guide on Multilingual Speech Recognition

This notebook presents a complete, self-contained implementation for fine-tuning OpenAI's Whisper model for Japanese Automatic Speech Recognition (ASR). We'll explore how Whisper's multilingual capabilities, acquired through pre-training on 680,000 hours of labeled audio data, can be adapted for Japanese language transcription through efficient fine-tuning 
techniques.

### Learning Objectives

- Configure Whisper for Japanese language transcription
- Apply LoRA to reduce trainable parameters by 98%
- Evaluate model performance using CER and WER metrics
- Deploy the fine-tuned model for inference

### The Challenge of Japanese ASR

Japanese automatic speech recognition presents unique challenges that distinguish it from languages like English:

1. **Writing System Complexity**: Japanese uses three distinct writing systems (Hiragana, Katakana, and Kanji) often mixed within single sentences
2. **No Natural Word Boundaries**: Unlike space-delimited languages, Japanese text flows continuously, making word segmentation non-trivial
3. **Context-Dependent Readings**: The same Kanji characters can have multiple pronunciations depending on context
4. **Homophones**: Japanese has numerous words that sound identical but have different meanings and written forms

Japanese ASR presents unique challenges that require specialized handling:

| Challenge | Description | Solution |
|-----------|-------------|----------|
| No word boundaries | Japanese text lacks spaces | Character-level evaluation (CER) |
| Three writing systems | Hiragana, Katakana, Kanji | Unicode normalization |
| Context-dependent readings | Same Kanji, different pronunciations | Large training corpus |

### Why Whisper?

Whisper, released by OpenAI in [September 2022](https://cdn.openai.com/papers/whisper.pdf), represents a paradigm shift in ASR systems. Unlike previous models like Wav2Vec 2.0 that rely on unsupervised pre-training, Whisper was trained on **680,000 hours of labeled audio-transcription data**, including 117,000 hours of multilingual content covering 96+ languages.

### Our Approach

In this workshop, we'll fine-tune the `whisper-base` model (74M parameters) for Japanese transcription. Through fine-tuning on the Common Voice dataset, we'll demonstrate how to:
- Adapt Whisper's multilingual knowledge to Japanese-specific patterns
- Optimize for Character Error Rate (CER) rather than Word Error Rate (WER)
- Achieve significant performance improvements with 2000 training samples

The complete training pipeline will take approximately 20-30 minutes on a GPU-enabled environment (A10G or similar).

## Workflow Overview

```mermaid
flowchart LR
    A[Audio Input] --> B[Feature Extraction]
    B --> C[Whisper Encoder]
    C --> D[LoRA Adapters]
    D --> E[Whisper Decoder]
    E --> F[Japanese Text]
    
    G[Training Data] --> H[Preprocessing]
    H --> B
```

### Model Specifications

We'll use the `whisper-base` configuration for this workshop:

| Model | Parameters | Layers | Width | Heads | Relative Speed |
|-------|------------|--------|-------|-------|----------------|
| tiny  | 39M        | 4      | 384   | 6     | ~10x           |
| **base** | **74M** | **6**  | **512** | **8** | **~7x**     |
| small | 244M       | 12     | 768   | 12    | ~3x            |
| medium| 769M       | 24     | 1024  | 16    | ~1x            |
| large | 1550M      | 32     | 1280  | 20    | ~0.6x          |

The base model offers an optimal balance between performance and computational efficiency for workshop environments.

## Environment Setup

We'll begin by configuring our environment and verifying GPU availability. A CUDA-enabled GPU significantly accelerates training - reducing time from hours to minutes.

In [ ]:
import os
import warnings
import torch
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import re
import unicodedata

# Configure environment
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Verify compute capabilities
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    gpu_properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_properties.name} | Memory: {gpu_properties.total_memory / 1024**3:.1f}GB | Compute: {gpu_properties.major}.{gpu_properties.minor}")
else:
    print("CPU mode - Training will be significantly slower")

## Dependencies Installation

The next cell installs all required packages from the project's pyproject.toml file. This ensures version compatibility across all dependencies.

**Note**: If you encounter installation errors:
1. Ensure you're in the correct directory relative to pyproject.toml
2. The installation may take a few minutes as it includes PyTorch and other large packages
3. Restart the kernel after installation if you see import errors

In [ ]:
# This installs all ASR-specific dependencies
!pip install -q -e ../../[asr]

## Import Libraries

In [ ]:
from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from jiwer import wer, cer
from huggingface_hub import login

## Data Processing Pipeline

### Understanding Japanese Text Normalization

Japanese text normalization is crucial for accurate ASR evaluation. The complexity arises from:

1. **Multiple character encodings**: Full-width vs half-width characters (ｱ vs ア)
2. **Punctuation variations**: Japanese-specific punctuation marks that don't affect pronunciation
3. **Unicode normalization**: NFKC normalization ensures consistent character representation

Our normalization function handles these challenges to ensure fair metric computation between predictions and ground truth.

In [ ]:
def normalize_japanese_text(text):
    """
    Normalize Japanese text for fair ASR evaluation.
    Handles full-width/half-width conversion and removes punctuation.
    """
    # Convert to NFKC form (normalizes full-width to half-width)
    text = unicodedata.normalize("NFKC", text)
    
    # Remove common Japanese punctuation that doesn't affect pronunciation
    punctuation = '。、！？「」『』（）【】〈〉《》・…〜'
    for p in punctuation:
        text = text.replace(p, '')
    
    # Remove extra spaces and convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip().lower()
    
    return text

def prepare_dataset(batch):
    """
    Prepare dataset for training by extracting features and encoding labels.
    """
    # Load and resample audio
    audio = batch["audio"]
    
    # Extract input features
    batch["input_features"] = feature_extractor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # Encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    
    return batch

## Data Collator

The data collator handles dynamic padding and prepares batches for training.

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data collator for speech-to-text models.
    Handles padding of input features and labels.
    """
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate input features and labels
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        # Pad input features
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        # IMPORTANT: Match dtype with model on GPU
        # The model is loaded as float16 on GPU, so we need to match that dtype
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if device.type == "cuda":
            batch["input_features"] = batch["input_features"].to(dtype=torch.float16)
        
        # Pad labels
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        # Replace padding token id with -100 for loss computation
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        # Remove bos token if present
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        return batch

## Metrics Computation

Define functions to compute Word Error Rate (WER) and Character Error Rate (CER).

In [ ]:
def compute_metrics(eval_pred):
    """
    Compute WER and CER metrics for evaluation.
    CER is more meaningful for Japanese due to lack of word boundaries.
    """
    pred_ids = eval_pred.predictions
    label_ids = eval_pred.label_ids
    
    # Replace -100 with pad token id
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    
    # Decode predictions and labels
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    # Normalize text for fair comparison
    pred_str = [normalize_japanese_text(text) for text in pred_str]
    label_str = [normalize_japanese_text(text) for text in label_str]
    
    # Calculate error rates
    wer_score = 100 * wer(label_str, pred_str)
    cer_score = 100 * cer(label_str, pred_str)
    
    return {"wer": wer_score, "cer": cer_score}

## Initialize Model and Processors

In [ ]:
# Suppress warnings at module import level
import warnings
import logging
warnings.filterwarnings("ignore", message="Both `max_new_tokens`")
logging.getLogger("transformers.generation.configuration_utils").setLevel(logging.ERROR)

# Load feature extractor
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-base")

# Load tokenizer with explicit settings
tokenizer = WhisperTokenizer.from_pretrained(
    "openai/whisper-base",
    language="japanese",
    task="transcribe"
)

# Load processor (combines feature extractor and tokenizer)
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-base",
    language="japanese",
    task="transcribe"
)

# Load model with BF16 dtype for A10G GPU
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-base",
    torch_dtype=torch.bfloat16  # A10G supports BF16
)

# Configure model generation settings for Japanese
model.generation_config.language = "japanese"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="japanese",
    task="transcribe"
)

# Generation config - keep max_length for trainer compatibility
model.generation_config.max_length = 448  
model.generation_config.max_new_tokens = 80  # Actual generation limit
model.generation_config.min_length = 1
model.generation_config.num_beams = 1
model.generation_config.do_sample = False
model.generation_config.temperature = 1.0
model.generation_config.repetition_penalty = 1.0
model.generation_config.length_penalty = 1.0
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.early_stopping = True
model.generation_config.use_cache = True

# Move model to device
model = model.to(device)

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()

## Load and Prepare Dataset

We'll use Mozilla Common Voice dataset for Japanese.

In [ ]:
# Authenticate with HuggingFace
HF_TOKEN = "HF-API-TOKEN"  # Replace with your token
login(token=HF_TOKEN, add_to_git_credential=False)

train_size = 2000  # Increased for meaningful training (20 mins on A10G)
test_size = 200    # 10% of training for validation

print(f"Loading Common Voice dataset: {train_size} train, {test_size} test samples")

dataset = DatasetDict()
dataset["train"] = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    "ja",
    split=f"train[:{train_size}]",
    token=HF_TOKEN,
    trust_remote_code=True
)
dataset["test"] = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    "ja",
    split=f"test[:{test_size}]",
    token=HF_TOKEN,
    trust_remote_code=True
)

# Keep only required columns
dataset = dataset.select_columns(["audio", "sentence"])

# Resample audio to 16kHz (Whisper's expected sample rate)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print(f"Dataset loaded: {len(dataset['train'])} train, {len(dataset['test'])} test samples")

## Dataset Exploration & Visualization

Before processing the dataset, let's explore its characteristics to better understand the distribution of audio lengths, text lengths, and Japanese character usage. These insights help us optimize training parameters and understand potential challenges.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare for visualization (2x2 grid, but only using 3)
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Japanese ASR Training Dataset Analysis', fontsize=16, y=1.02)

# 1. Audio Duration Distribution
print("Analyzing dataset characteristics...")
audio_durations = []
for sample in dataset["train"]:
    # Calculate duration in seconds
    audio_array = sample["audio"]["array"]
    sample_rate = sample["audio"]["sampling_rate"]
    duration = len(audio_array) / sample_rate
    audio_durations.append(duration)

ax1.hist(audio_durations, bins=30, color='#2ca02c', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Duration (seconds)')
ax1.set_ylabel('Count')
ax1.set_title(f'Audio Duration Distribution (n={len(audio_durations)})')
ax1.axvline(np.mean(audio_durations), color='red', linestyle='--', 
            label=f'Mean: {np.mean(audio_durations):.1f}s')
ax1.axvline(np.median(audio_durations), color='orange', linestyle='--', 
            label=f'Median: {np.median(audio_durations):.1f}s')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Sample Rate Verification (fixed to use proper indexing)
print("Verifying sample rates...")
sample_rates = []
# Only check first 100 samples
for i, sample in enumerate(dataset["train"]):
    if i >= 100:
        break
    sample_rates.append(sample["audio"]["sampling_rate"])

unique_rates, counts = np.unique(sample_rates, return_counts=True)
bars = ax2.bar(range(len(unique_rates)), counts, color='#1f77b4', alpha=0.7)
ax2.set_xticks(range(len(unique_rates)))
ax2.set_xticklabels([f'{rate}Hz' for rate in unique_rates])
ax2.set_ylabel('Count')
ax2.set_title('Sample Rate Distribution (Expected: 16kHz)')
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, count in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', va='bottom')

# 3. Text Length Distribution
text_lengths = []
for sample in dataset["train"]:
    text = sample["sentence"]
    text_lengths.append(len(text))

ax3.hist(text_lengths, bins=30, color='#ff7f0e', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Text Length (characters)')
ax3.set_ylabel('Count')
ax3.set_title(f'Japanese Text Length Distribution')
ax3.axvline(np.mean(text_lengths), color='red', linestyle='--',
            label=f'Mean: {np.mean(text_lengths):.0f} chars')
ax3.axvline(np.median(text_lengths), color='orange', linestyle='--',
            label=f'Median: {np.median(text_lengths):.0f} chars')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Duration vs Text Length Scatter Plot (instead of character analysis)
ax4.scatter(audio_durations, text_lengths, alpha=0.5, color='#9467bd')
ax4.set_xlabel('Audio Duration (seconds)')
ax4.set_ylabel('Text Length (characters)')
ax4.set_title('Audio Duration vs Text Length Correlation')
ax4.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(audio_durations, text_lengths, 1)
p = np.poly1d(z)
ax4.plot(sorted(audio_durations), p(sorted(audio_durations)), 
         "r--", alpha=0.7, label=f'Trend line')
ax4.legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print("=" * 50)
print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"\nAudio Duration:")
print(f"  Min: {np.min(audio_durations):.1f}s | Max: {np.max(audio_durations):.1f}s")
print(f"  Mean: {np.mean(audio_durations):.1f}s | Std: {np.std(audio_durations):.1f}s")
print(f"\nText Length:")
print(f"  Min: {np.min(text_lengths)} chars | Max: {np.max(text_lengths)} chars")
print(f"  Mean: {np.mean(text_lengths):.0f} chars | Std: {np.std(text_lengths):.0f} chars")
print(f"\nSample Rate: All samples at {unique_rates[0]}Hz" if len(unique_rates) == 1 else f"Multiple rates detected: {unique_rates}")

# Show 3 example samples
print("\n" + "=" * 50)
print(" Sample Examples from Training Dataset:")
print("=" * 50)

# Select 3 diverse samples based on text length (short, medium, long)
sorted_indices = np.argsort(text_lengths)
sample_indices = [
    sorted_indices[len(sorted_indices)//6],    # Short example (~16th percentile)
    sorted_indices[len(sorted_indices)//2],     # Medium example (median)
    sorted_indices[len(sorted_indices)*5//6]    # Long example (~83rd percentile)
]

for i, idx in enumerate(sample_indices, 1):
    sample = dataset["train"][int(idx)]
    duration = len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
    text = sample["sentence"]
    
    print(f"\n🎯 Example {i}:")
    print(f"  Duration: {duration:.2f} seconds")
    print(f"  Text length: {len(text)} characters")
    print(f"  Transcript: {text}")
    print(f"  Audio shape: {len(sample['audio']['array']):,} samples @ {sample['audio']['sampling_rate']}Hz")

## Process Dataset

Extract features and prepare the dataset for training.

In [ ]:
# Process dataset with prepare_dataset
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names["train"],
    num_proc=1,
    desc="Processing audio samples"
)

# Verify dataset structure
sample = dataset["train"][0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Input features shape: {len(sample['input_features'])} x {len(sample['input_features'][0])}")
print(f"Labels length: {len(sample['labels'])}")

## Configure Training

Set up training arguments with optimized hyperparameters.

In [ ]:
# Suppress generation warnings globally
import logging
import warnings
warnings.filterwarnings("ignore", message="Both `max_new_tokens`")
logging.getLogger("transformers.generation_utils").setLevel(logging.ERROR)
logging.getLogger("transformers.generation.configuration_utils").setLevel(logging.ERROR)

# Always use BF16 on this instance since it's supported
use_bf16 = True
print("Using BF16 mixed precision training (optimal for A10G)")

# Define output directory
output_dir = "./whisper-japanese-finetuned"
os.makedirs(output_dir, exist_ok=True)

# Training configuration
train_size = len(dataset["train"])
batch_size = 8
gradient_accumulation_steps = 2
steps_per_epoch = train_size // (batch_size * gradient_accumulation_steps)
num_epochs = 4
total_steps = steps_per_epoch * num_epochs


eval_steps = 25
save_steps = 50

# Training arguments with BF16
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=3e-5,
    warmup_steps=100,
    num_train_epochs=num_epochs,
    gradient_checkpointing=True,
    bf16=use_bf16,  # Always use BF16
    fp16=False,  # Don't use FP16
    evaluation_strategy="steps",
    eval_steps=eval_steps,
    save_steps=save_steps,
    save_total_limit=3,
    logging_steps=10,
    predict_with_generate=True,
    generation_num_beams=1,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
    weight_decay=0.01,
    optim="adamw_torch",
    max_grad_norm=1.0,
    lr_scheduler_type="linear",
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    label_names=["labels"],
    logging_nan_inf_filter=False,  
    seed=42,
    per_device_eval_batch_size=16,
    bf16_full_eval=False,  
)

# Create data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.tokenizer,
)

## Baseline Evaluation

Before fine-tuning, we'll establish baseline metrics to quantify the improvement our training provides. The pre-trained Whisper model, while multilingual, hasn't been optimized specifically for Japanese, so we expect high initial error rates.

### Understanding Expected Baseline Performance

For Japanese ASR on the pre-trained model:
- **CER (Character Error Rate)**: Typically 150-250% due to language mismatch
- **WER (Word Error Rate)**: Often exceeds 500% as word boundaries are misidentified

These high initial error rates are normal and demonstrate the necessity of fine-tuning for target languages.

In [ ]:
# Evaluate baseline performance
baseline_metrics = trainer.evaluate()

baseline_cer = baseline_metrics.get("eval_cer", 100)
baseline_wer = baseline_metrics.get("eval_wer", 100)

print(f"Baseline CER: {baseline_cer:.2f}% | WER: {baseline_wer:.2f}%")

## Fine-Tuning Process

### Training Strategy

Our fine-tuning approach employs several optimization techniques:

1. **Gradient Checkpointing**: Trades computation for memory, enabling larger batch sizes
2. **Mixed Precision Training**: Uses BF16/FP16 to accelerate training while maintaining stability
3. **Weight Decay**: L2 regularization to prevent overfitting on limited data
4. **Warmup Schedule**: Gradual learning rate increase to stabilize early training

### What Happens During Training

The model will:
- Process audio through the frozen encoder (or fine-tune if not using LoRA)
- Learn Japanese-specific patterns in the decoder
- Optimize the cross-attention mechanism for Japanese acoustics
- Save checkpoints every 100 iterations to output directory

Training duration depends on:
- GPU model: T4 (~30 min), V100 (~15 min), A100 (~10 min)
- Dataset size: Linear scaling with number of samples
- Batch size: Larger batches reduce training time but require more memory

In [ ]:
# Launch training
train_result = trainer.train()

# Save the final model
trainer.save_model()
processor.save_pretrained(output_dir)

print(f"Training complete. Final loss: {train_result.metrics.get('train_loss', 'N/A'):.4f}")
print(f"Model saved to: {os.path.abspath(output_dir)}")

In [ ]:
# Visualize training metrics
import matplotlib.pyplot as plt

# Get training history from trainer
history = trainer.state.log_history

# Extract metrics safely
train_loss = [x['loss'] for x in history if 'loss' in x]
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
eval_cer = [x['eval_cer'] for x in history if 'eval_cer' in x]
steps_train = [x['step'] for x in history if 'loss' in x]
steps_eval = [x['step'] for x in history if 'eval_loss' in x]

# Only create plots if we have data
if train_loss:
    fig, axes = plt.subplots(1, 2 if eval_loss else 1, figsize=(14 if eval_loss else 7, 5))
    
    # Ensure axes is always a list for consistent handling
    if not eval_loss:
        axes = [axes]
    else:
        axes = list(axes)
    
    # Plot 1: Training loss (always available)
    axes[0].plot(steps_train, train_loss, label='Training Loss', color='blue', alpha=0.7)
    if eval_loss:
        axes[0].plot(steps_eval, eval_loss, label='Validation Loss', color='red', marker='o')
    axes[0].set_xlabel('Steps')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: CER over time (only if evaluation data exists)
    if eval_loss and len(axes) > 1:
        axes[1].plot(steps_eval, eval_cer, label='CER', color='green', marker='o')
        axes[1].axhline(y=baseline_cer, color='orange', linestyle='--', 
                       label=f'Baseline CER ({baseline_cer:.1f}%)')
        axes[1].set_xlabel('Steps')
        axes[1].set_ylabel('Character Error Rate (%)')
        axes[1].set_title('CER During Training')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('Training Progress', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print available metrics
    print(f"\n📊 Training Summary:")
    print("=" * 50)
    if train_loss:
        print(f"Final training loss: {train_loss[-1]:.4f}")
        print(f"Initial training loss: {train_loss[0]:.4f}")
        print(f"Loss reduction: {(train_loss[0] - train_loss[-1]):.4f}")
    
    if eval_loss:
        print(f"\nFinal validation loss: {eval_loss[-1]:.4f}")
        print(f"Final CER: {eval_cer[-1]:.2f}%")
        print(f"Best CER achieved: {min(eval_cer):.2f}% at step {steps_eval[eval_cer.index(min(eval_cer))]}")
    else:
        print("\n⚠️ Note: No evaluation metrics recorded during training.")
        print(f"   Training steps: {steps_train[-1] if steps_train else 0}, Eval frequency: every {trainer.args.eval_steps} steps")
else:
    print("No training metrics available to visualize.")

## Post-Training Evaluation

After fine-tuning, we'll evaluate the model's performance and compare it with the baseline. The metrics will reveal how effectively the model has adapted to Japanese speech patterns.

In [ ]:
# Evaluate fine-tuned model
final_metrics = trainer.evaluate()

final_cer = final_metrics.get("eval_cer", 100)
final_wer = final_metrics.get("eval_wer", 100)

# Calculate improvements
cer_improvement = baseline_cer - final_cer
wer_improvement = baseline_wer - final_wer
cer_relative = (cer_improvement / baseline_cer) * 100 if baseline_cer > 0 else 0
wer_relative = (wer_improvement / baseline_wer) * 100 if baseline_wer > 0 else 0

# Display results
print(f"{'Metric':<20} {'Baseline':>10} {'Fine-tuned':>12} {'Improvement':>12}")
print("-" * 55)
print(f"{'CER (%)':<20} {baseline_cer:>10.2f} {final_cer:>12.2f} {cer_improvement:>12.2f}")
print(f"{'WER (%)':<20} {baseline_wer:>10.2f} {final_wer:>12.2f} {wer_improvement:>12.2f}")
print(f"{'Relative CER Δ (%)':<20} {cer_relative:>45.1f}")
print(f"{'Relative WER Δ (%)':<20} {wer_relative:>45.1f}")

In [ ]:
# Create performance comparison visualization
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Chart 1: Before/After Comparison
metrics = ['CER', 'WER']
baseline_values = [baseline_cer, baseline_wer]
finetuned_values = [final_cer, final_wer]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax1.bar(x - width/2, baseline_values, width, label='Baseline', color='#ff7f0e')
bars2 = ax1.bar(x + width/2, finetuned_values, width, label='Fine-tuned', color='#2ca02c')

ax1.set_ylabel('Error Rate (%)')
ax1.set_title('Performance Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

# Chart 2: Relative Improvements
improvements = [cer_relative, wer_relative]
colors = ['#2ca02c' if imp > 0 else '#ff7f0e' for imp in improvements]
bars = ax2.bar(metrics, improvements, color=colors, alpha=0.7)

ax2.set_ylabel('Relative Improvement (%)')
ax2.set_title('Performance Gains')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.grid(axis='y', alpha=0.3)

# Add percentage labels
for bar, value in zip(bars, improvements):
    ax2.annotate(f'{value:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, value),
                xytext=(0, 3 if value > 0 else -15),
                textcoords="offset points",
                ha='center', va='bottom' if value > 0 else 'top')

plt.suptitle('Whisper Fine-tuning Results for Japanese ASR', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Model Inference

With our fine-tuned model complete, we can now use it for Japanese speech transcription. The model can be loaded from the saved checkpoint for inference on new audio data.

In [ ]:
def transcribe_audio(audio_path: str) -> str:
    """
    Transcribe an audio file using the fine-tuned model.
    
    Args:
        audio_path: Path to the audio file (WAV, MP3, etc.)
    
    Returns:
        Transcribed Japanese text
    """
    from transformers import pipeline
    
    # Create ASR pipeline with our fine-tuned model
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        max_new_tokens=128,
        generate_kwargs={"language": "japanese", "task": "transcribe"},
        device=0 if device.type == "cuda" else -1
    )
    
    # Perform transcription
    result = pipe(audio_path)
    return result["text"]

# Usage example
print(f"Fine-tuned model location: {os.path.abspath(output_dir)}")
print(f"\nTo transcribe audio: transcription = transcribe_audio('path/to/audio.wav')")

## Conclusion

In this workshop, we've successfully fine-tuned Whisper model for Japanese ASR. The key achievements include:

### Key Learnings
1. **Data Quality > Quantity**: Even 500 samples can provide improvements, though 1000+ is recommended
2. **Hyperparameter Sensitivity**: Learning rate and batch size critically impact convergence
3. **Evaluation Metrics**: CER is more meaningful than WER for continuous-script languages
4. **Mixed Precision**: BF16 provides better stability than FP16 for transformer training